# Elife Lung Cancer Non Coding Biomarker Random Forest
Andrew E. Davidson aedaivds@ucsc.edu 03/04/25

Copyright (c) 2020-2023, Regents of the University of California All rights reserved. https://polyformproject.org/licenses/noncommercial/1.0.0

1. Train a Random Forest using all the non-coding genes from the Lung Cancer', 'Healthy donor' samples
2. Use feature Importance to identify set of best features


ref:  
- intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/createNonCodingDataSet.ipynb
- intraExtraRNA_POC/jupyterNotebooks/elife/elifeBinaryRandomForestResults.ipynb

In [1]:
import ipynbname

# use display() to print an html version of a data frame
# useful if dataFrame output is not generated by last like of cell
from IPython.display import display

import joblib
import math
import numpy as np
import os
import pandas as pd
import pprint as pp
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import sys

notebookName = ipynbname.name()
notebookPath = ipynbname.path()
notebookDir = os.path.dirname(notebookPath)

#outDir = f'{notebookDir}/{notebookName}.out'
outDir = f'/private/groups/kimlab/aedavids/elife/{notebookName}.out'
os.makedirs(outDir, exist_ok=True)
print(f'outDir:\n{outDir}')

dataOutDir = os.path.join(outDir, "data")
os.makedirs(dataOutDir, exist_ok=True)
print(f'\ndataOutDir ;\n{dataOutDir}')

import logging
#loglevel = "DEBUG"
#loglevel = "INFO"
loglevel = "WARN"
# logFMT = "%(asctime)s %(levelname)s [thr:%(threadName)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logFMT = "%(asctime)s %(levelname)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logging.basicConfig(format=logFMT, level=loglevel)    
logger = logging.getLogger(notebookName)

meaningOfLife = 42

outDir:
/private/groups/kimlab/aedavids/elife/elifeLungCancerNon-CodingBiomarkerRandomForest.out

dataOutDir ;
/private/groups/kimlab/aedavids/elife/elifeLungCancerNon-CodingBiomarkerRandomForest.out/data


In [2]:
# setting the python path allows us to run python scripts from using
# the CLI. 
ORIG_PYTHONPATH = os.environ['PYTHONPATH']

deconvolutionModules = notebookPath.parent.joinpath("../../../../deconvolutionAnalysis/python/")
print("deconvolutionModules: {}\n".format(deconvolutionModules))

PYTHONPATH = ORIG_PYTHONPATH + f':{deconvolutionModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

intraExtraRNA_POCModules=notebookPath.parent.joinpath("../../../python/src")
print("intraExtraRNA_POCModules: {}\n".format(intraExtraRNA_POCModules))

PYTHONPATH = PYTHONPATH + f':{intraExtraRNA_POCModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

########
os.environ["PYTHONPATH"] = PYTHONPATH
PYTHONPATH = os.environ["PYTHONPATH"]
print("PYTHONPATH: {}\n".format(PYTHONPATH))

# to be able to import our local python files we need to set the sys.path
# https://stackoverflow.com/a/50155834
sys.path.append( str(deconvolutionModules) )
sys.path.append( str(intraExtraRNA_POCModules) )
#print("\nsys.path:\n{}\n".format(sys.path))

deconvolutionModules: /private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../../deconvolutionAnalysis/python

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../../deconvolutionAnalysis/python

intraExtraRNA_POCModules: /private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../../deconvolutionAnalysis/python:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../../deconvolutionAnalysis/python:/private/home/aedavid

In [3]:
from intraExtraRNA.elifeUtilities import loadMetaData

# Data Prep
Load count and meta data.  
Fit a Label Encoder

In [4]:
annotated_elife_lung_norm_counts_path = "/private/groups/kimlab/aedavids/elife/createNonCodingDataSet.out/data/annotated_elife_lung_norm_counts_2023-05-18.csv"
annotatedDF = pd.read_csv( annotated_elife_lung_norm_counts_path, index_col='gene')

In [5]:
print( annotatedDF.shape )
annotatedDF.iloc[0:5, 0:5]

(56607, 79)


,gene_biotype,SRR14506690,SRR14506691,SRR14506692,SRR14506693
gene,,,,,
(A)n,Microsatellite,148.057125,121.600515,551.413204,45.545968
(AAA)n,Microsatellite,0.000000,0.000000,0.000000,0.000000
(AAAAAAC)n,Microsatellite,0.000000,0.000000,0.000000,0.000000
(AAAAAAG)n,Microsatellite,0.000000,0.000000,0.000000,0.000000
(AAAAAAT)n,Microsatellite,0.000000,0.000000,0.000000,0.000000


In [6]:
byCol = 1
elifeLungCountsDF = annotatedDF.drop('gene_biotype', axis=byCol)
print( elifeLungCountsDF.shape )
#elifeLungCountsDF.iloc[0:5, 0:5]

(56607, 78)


In [7]:
# random forest fit expects count data to be samples x genes
elifeLungCountsDF = elifeLungCountsDF.transpose()
print(elifeLungCountsDF.shape)
elifeLungCountsDF.iloc[0:5, 0:5]

(78, 56607)


gene,(A)n,(AAA)n,(AAAAAAC)n,(AAAAAAG)n,(AAAAAAT)n
SRR14506690,148.057125,0.0,0.0,0.0,0.0
SRR14506691,121.600515,0.0,0.0,0.0,0.0
SRR14506692,551.413204,0.0,0.0,0.0,0.0
SRR14506693,45.545968,0.0,0.0,0.0,0.0
SRR14506694,86.068765,0.0,0.0,0.0,0.0


In [8]:
elifeMetaDF = loadMetaData()
selelectLungRows = elifeMetaDF['diagnosis'].isin( ['Lung Cancer', 'Healthy donor'] )
lungSampleIdDF = elifeMetaDF.loc[ selelectLungRows, :]
lungSampleIdDF

,sample_id,diagnosis
31,SRR14506690,Lung Cancer
32,SRR14506691,Lung Cancer
33,SRR14506692,Lung Cancer
34,SRR14506693,Lung Cancer
35,SRR14506694,Lung Cancer
...,...,...
219,SRR14506884,Healthy donor
220,SRR14506885,Healthy donor
221,SRR14506886,Healthy donor
222,SRR14506887,Healthy donor


In [9]:
# we can not control which class is labled zero
labelEncoder = LabelEncoder()
labelEncoder.fit( lungSampleIdDF.loc[:, 'diagnosis'])
print( labelEncoder.classes_)
yNP = labelEncoder.transform( lungSampleIdDF.loc[:, 'diagnosis'] )
yNP

['Healthy donor' 'Lung Cancer']


array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

# Train Model

In [10]:
%%time
rfModel = RandomForestClassifier(random_state=meaningOfLife) # , **kwags
rfModel.fit(elifeLungCountsDF, yNP)

CPU times: user 509 ms, sys: 3.95 ms, total: 513 ms
Wall time: 512 ms


RandomForestClassifier(random_state=42)

# Evaluate Model

In [11]:
def selectImportantFeatures( model : RandomForestClassifier) -> pd.DataFrame:
    '''
    returns a data frame with features that have non zero importance
    sorted in descending order by importance

    importance is a value between 0 and 1. The values are normalize

    example output
                      name  importance
    451  ENSG00000277194.1    0.014160
    489  ENSG00000285909.1    0.011819
    '''
    # create a knock out vector
    nonZero = rfModel.feature_importances_ > 0.0
    
    importances = rfModel.feature_importances_[nonZero]
    featureNames = elifeLungCountsDF.columns[nonZero]
    retDF = pd.DataFrame( {
                            'name' : featureNames,
                            'importance' : importances
                        })
    
    retDF.sort_values(by="importance", ascending=False, inplace=True)

    return retDF

###################
featureImportanceDF = selectImportantFeatures( rfModel )
print('featureImportanceDF.shape :', featureImportanceDF.shape)
featureImportanceDF.head()

featureImportanceDF.shape : (631, 2)


,name,importance
451,ENSG00000277194.1,0.014160
489,ENSG00000285909.1,0.011819
542,ENSG00000289601.1,0.010526
231,ENSG00000237550.6,0.010144
381,ENSG00000264769.1,0.009759


In [12]:
# explore the gene_biotypes of important features
selectImportantRows = annotatedDF.index.isin( featureImportanceDF.loc[:, 'name'] )
importGenBioTypeSeries = annotatedDF.loc[selectImportantRows, 'gene_biotype']
print(importGenBioTypeSeries.shape)
importGenBioTypeSeries.groupby(importGenBioTypeSeries).count()

(631,)


gene_biotype
DNA                10
LINE               15
LTR                48
Microsatellite     26
Other             204
SINE                7
lncRNA            321
Name: gene_biotype, dtype: int64

In [13]:
def calculateAccuracy( 
            rfModel : RandomForestClassifier,
            xDF : pd.DataFrame,
            yNP : np.array ) :
    '''
    todo
    '''
    predictions = rfModel.predict( xDF )
    n = len(yNP)
    accuracy = sum( predictions == yNP ) / n

    return accuracy
    


accuracy = calculateAccuracy(rfModel , elifeLungCountsDF, yNP)
print( f'accuracy : {accuracy * 100}%' )

accuracy : 100.0%
